In [1]:
%matplotlib inline

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

In [3]:
hfdataset= HFDatasetLoader(dataset_name="aptologia/precios-arriendo-colombia", split="train", filename="datos/precios-arriendo-colombia_clasificacion.csv")

In [4]:
df_clasificacion_crudo = hfdataset.load()
df_clasificacion_crudo.head(2)

,fecha,ciudad_id,ciudad,barrio,avisos,p25,mediana,p75,con_parqueadero,con_telefono
0,2026-10-05,medellin,Medellín,El Poblado,17124,4400000,5700000,7700000,15907,6599
1,2026-10-05,medellin,Medellín,Poblado,7593,4100000,5650000,8000000,6509,841


# Comvierto mi database para poder tener informacion de texto agregando una columna que me indique si el valor de la propiedad entra en una categoria de alto medio o bajo.

In [5]:
import numpy as np
import pandas as pd

def crear_texto_propiedad(row):
  tokens = []

  # Categóricas directas
  tokens.append(f"ciudad_{row['ciudad_id']}")
  tokens.append(f"barrio_{str(row['barrio']).lower().replace(' ', '_')}")

  # Rango de precios basado en la mediana
  mediana = row["mediana"]
  if mediana >= 7000000:
    tokens.append("precio_alto_gama_alta")
  elif mediana >= 4500000:
    tokens.append("precio_medio")
  else:
    tokens.append("precio_economico")

  # Proporción de equipamiento (Parqueadero y Teléfono)
  if row["avisos"] > 0:
    prop_parq = row["con_parqueadero"] / row["avisos"]
    prop_tel = row["con_telefono"] / row["avisos"]

    if prop_parq > 0.7:
      tokens.append("alto_parqueadero")
    if prop_tel > 0.5:
      tokens.append("alta_conectividad_telefono")

  # Nivel de volumen de oferta
  if row["avisos"] > 10000:
    tokens.append("alta_oferta_avisos")

  return " ".join(tokens)

# Aplicar la transformación a tu dataframe
dfc_clasificacion_crudo = df_clasificacion_crudo.copy()
dfc_clasificacion_crudo["text"] = dfc_clasificacion_crudo.apply(crear_texto_propiedad, axis=1)

In [6]:
dfc_clasificacion_crudo.head(2)


,fecha,ciudad_id,ciudad,barrio,avisos,p25,mediana,p75,con_parqueadero,con_telefono,text
0,2026-10-05,medellin,Medellín,El Poblado,17124,4400000,5700000,7700000,15907,6599,ciudad_medellin barrio_el_poblado precio_medio...
1,2026-10-05,medellin,Medellín,Poblado,7593,4100000,5650000,8000000,6509,841,ciudad_medellin barrio_poblado precio_medio al...


## Clasificacion

In [8]:
pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve, precision_score, recall_score

from clases_po import ClasificadorTexto   # versión con alias de columnas y palabras_clave()

CLASES = ["alto", "medio", "economico"]
COLORES = {"medio": "#1f6fe5", "alto": "#f2661b", "economico": "#e5384b"}
UMBRAL_COSTOSO = 0.5                                  # ajústalo después de ver el bloque 1
CACHE_COSTOSO = Path("datos/precios-arriendo-colombia.csv")  # el modelo de COSTOSIDAD corre una sola vez
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

df = dfc_clasificacion_crudo.copy()
df.loc[:, "text"] = df["text"].fillna("").astype(str).str.strip()

In [8]:
dfc_clasificacion_crudo.head(5)

,fecha,ciudad_id,ciudad,barrio,avisos,p25,mediana,p75,con_parqueadero,con_telefono,text
0,2026-10-05,medellin,Medellín,El Poblado,17124,4400000,5700000,7700000,15907,6599,ciudad_medellin barrio_el_poblado precio_medio...
1,2026-10-05,medellin,Medellín,Poblado,7593,4100000,5650000,8000000,6509,841,ciudad_medellin barrio_poblado precio_medio al...
2,2026-10-05,medellin,Medellín,Laureles,5780,3300000,4200000,5300000,4283,2169,ciudad_medellin barrio_laureles precio_economi...
3,2026-10-05,medellin,Medellín,Castropol,2730,4200000,5350000,7000000,2286,1547,ciudad_medellin barrio_castropol precio_medio ...
4,2026-10-05,medellin,Medellín,El Tesoro,2471,5800000,7700000,11500000,2167,1278,ciudad_medellin barrio_el_tesoro precio_alto_g...


In [9]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [10]:
%pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [11]:
df.head(5)

,fecha,ciudad_id,ciudad,barrio,avisos,p25,mediana,p75,con_parqueadero,con_telefono,text
0,2026-10-05,medellin,Medellín,El Poblado,17124,4400000,5700000,7700000,15907,6599,ciudad_medellin barrio_el_poblado precio_medio...
1,2026-10-05,medellin,Medellín,Poblado,7593,4100000,5650000,8000000,6509,841,ciudad_medellin barrio_poblado precio_medio al...
2,2026-10-05,medellin,Medellín,Laureles,5780,3300000,4200000,5300000,4283,2169,ciudad_medellin barrio_laureles precio_economi...
3,2026-10-05,medellin,Medellín,Castropol,2730,4200000,5350000,7000000,2286,1547,ciudad_medellin barrio_castropol precio_medio ...
4,2026-10-05,medellin,Medellín,El Tesoro,2471,5800000,7700000,11500000,2167,1278,ciudad_medellin barrio_el_tesoro precio_alto_g...


In [9]:
# 1. Definición de Constantes
CLASES = ["alto", "medio", "economico"]
COLORES = {"medio": "#1f6fe5", "alto": "#f2661b", "economico": "#e5384b"}
UMBRAL_COSTOSO = 0.5 

CACHE_COSTOSO = Path("datos/precios-arriendo-v2.csv")

# Aislamiento total del DataFrame
df = df.copy()

if "predio_id" not in df.columns:
    df.loc[:, "predio_id"] = df.index

# FIX LÍNEA 24: Asignación segura con .loc para evitar FutureWarning
df.loc[:, "text"] = df["text"].fillna("").astype(str)

# 2. Lógica del Modelo y Caché
if CACHE_COSTOSO.exists():
    prob_df = pd.read_csv(CACHE_COSTOSO)
    print("Probabilidades de precio alto leídas de", CACHE_COSTOSO)
else:
    print("Entrenando modelo de estimación de costos...")
    
    y_binary = (df["mediana"] >= 7000000).astype(int)
    
    vec = CountVectorizer()
    X = vec.fit_transform(df["text"])
    
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X, y_binary)
    
    prob_costoso = clf.predict_proba(X)[:, 1]
    
    prob_df = pd.DataFrame(
        {"predio_id": df["predio_id"].to_numpy(), "p_costoso": prob_costoso}
    )
    
    CACHE_COSTOSO.parent.mkdir(parents=True, exist_ok=True)
    prob_df.to_csv(CACHE_COSTOSO, index=False)
    print("Guardado en", CACHE_COSTOSO)

# 3. Integración de resultados
df = df.drop(columns=["p_costoso"], errors="ignore").merge(
    prob_df, on="predio_id", how="left"
)

# FIX LÍNEA 55: Asignación segura con .loc para evitar FutureWarning
df.loc[:, "p_costoso"] = df["p_costoso"].fillna(0.0)

# 4. Evaluación de Umbrales
print("\n--- Sensibilidad del umbral para clasificar como COSTOSO/ALTO ---")
for u in [0.3, 0.5, 0.7, 0.9]:
    total = (df["p_costoso"] >= u).sum()
    print(f"Umbral {u}: {total:>5} registros serían clasificados como 'alto'")

# 5. Histograma de Probabilidades
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(df["p_costoso"], bins=50, color=COLORES["alto"])
ax.axvline(
    UMBRAL_COSTOSO,
    color="#151b6e",
    ls="--",
    label=f"Umbral Costoso {UMBRAL_COSTOSO}",
)
ax.set_yscale("log")
ax.set_title("Distribución de P(Costoso) en los predios (Escala logarítmica)")
ax.set_xlabel("P(Costoso | Predio/Barrio)")
ax.set_ylabel("Frecuencia (log)")
ax.legend()
plt.tight_layout()
plt.show()

# 6. Asignación de Etiquetas Definitivas
condiciones = [
    df["p_costoso"] >= UMBRAL_COSTOSO, 
    (df["mediana"] < 4500000)
]
elecciones = ["alto", "economico"]
df.loc[:, "etiqueta"] = np.select(condiciones, elecciones, default="medio")

print("\n--- Conteo de Etiquetas Finales ---")
print(df["etiqueta"].value_counts())

# 7. Muestra de Predios clasificados como 'Alto'
print("\nEjemplos marcados con categoría ALTO:")
pd.set_option("display.max_colwidth", 160)
columnas_mostrar = ["ciudad", "barrio", "mediana", "p_costoso", "text"]
print(df[df["etiqueta"] == "alto"].nlargest(8, "p_costoso")[columnas_mostrar])

Probabilidades de precio alto leídas de datos/precios-arriendo-v2.csv

--- Sensibilidad del umbral para clasificar como COSTOSO/ALTO ---
Umbral 0.3:    85 registros serían clasificados como 'alto'
Umbral 0.5:    85 registros serían clasificados como 'alto'
Umbral 0.7:    85 registros serían clasificados como 'alto'
Umbral 0.9:    79 registros serían clasificados como 'alto'

--- Conteo de Etiquetas Finales ---
etiqueta
economico    1199
medio         176
alto           85
Name: count, dtype: int64

Ejemplos marcados con categoría ALTO:
     ciudad              barrio   mediana  p_costoso  \
508  Bogotá               Chico   7500000   0.960823   
510  Bogotá     Chico Reservado  10000000   0.960823   
512  Bogotá         Los Rosales  10500000   0.960823   
520  Bogotá            El Chico   7000000   0.960823   
532  Bogotá         La Carolina   8250000   0.960823   
537  Bogotá           El Virrey   8000000   0.960823   
538  Bogotá             Cabrera  15000000   0.960823   
547  Bogot

In [13]:
print("--- 2 · Estructura y calidad -> escena 6 ---")
print("Forma (filas, columnas):", df.shape)
print("\n--- Información del DataFrame ---")
df.info()

print("\n--- Valores faltantes por columna ---")
print(df.isna().sum())

print("\n--- Análisis de duplicados y vacíos ---")
# Verifica si hay combinaciones de texto idénticas (mismo barrio, ciudad y características)
print("Textos descriptivos duplicados exactos:", df.duplicated(subset=["text"]).sum())

# Verifica si algún registro quedó con la columna 'text' completamente vacía
print("Registros con texto vacío:", (df["text"].str.strip() == "").sum())

# Reemplazamos la revisión de 'message' por una revisión de 'barrio' (columna clave en tus datos)
print("Registros sin barrio especificado (nulos o en blanco):", 
      df["barrio"].isna().sum() + (df["barrio"].astype(str).str.strip() == "").sum())

--- 2 · Estructura y calidad -> escena 6 ---
Forma (filas, columnas): (1460, 14)

--- Información del DataFrame ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   fecha            1460 non-null   object 
 1   ciudad_id        1460 non-null   object 
 2   ciudad           1460 non-null   object 
 3   barrio           1460 non-null   object 
 4   avisos           1460 non-null   int64  
 5   p25              1460 non-null   int64  
 6   mediana          1460 non-null   int64  
 7   p75              1460 non-null   int64  
 8   con_parqueadero  1460 non-null   int64  
 9   con_telefono     1460 non-null   int64  
 10  text             1460 non-null   object 
 11  predio_id        1460 non-null   int64  
 12  p_costoso        1460 non-null   float64
 13  etiqueta         1460 non-null   object 
dtypes: float64(1), int64(7), object(6)
m

In [10]:
print("--- 3 · Balance de las tres clases -> escena 10 ---")

# Cálculo de frecuencias y porcentajes
conteo = df["etiqueta"].value_counts().reindex(CLASES).fillna(0).astype(int)
porc = (conteo / conteo.sum() * 100).round(2)
print(pd.DataFrame({"predios": conteo, "%": porc}))

# Gráfico de barras
fig, ax = plt.subplots(figsize=(6.5, 3.5))
barras = ax.bar(conteo.index, conteo.values, color=[COLORES.get(c, "gray") for c in conteo.index])
ax.bar_label(barras, labels=[f"{v:,}  ({p} %)" for v, p in zip(conteo.values, porc.values)])
ax.set_title("Distribución de Predios por Clase de Precio")
ax.set_ylabel("Cantidad de registros")
plt.tight_layout()
plt.show()

# Pere, el modelo perezoso (Baseline): siempre predice la clase mayoritaria
mayoritaria = conteo.idxmax()
print(f"Si el modelo perezoso responde siempre '{mayoritaria}': "
      f"exactitud = {conteo.max() / conteo.sum():.1%}, "
      f"predios de valor 'alto' detectados = 0 de {conteo.get('alto', 0)}")

--- 3 · Balance de las tres clases -> escena 10 ---
           predios      %
etiqueta                 
alto            85   5.82
medio          176  12.05
economico     1199  82.12
Si el modelo perezoso responde siempre 'economico': exactitud = 82.1%, predios de valor 'alto' detectados = 0 de 85


In [11]:
print("--- 4 · Longitud del texto descriptivo por clase -> escenas 5 y 6 ---")

# Solución al warning: aislar el DataFrame para trabajar de forma segura
df = df.copy()

# Contar la cantidad de palabras (tokens) por registro usando .loc
df.loc[:, "palabras"] = df["text"].str.split().str.len()

# Mostrar estadísticas descriptivas de la longitud por categoría
print(df.groupby("etiqueta")["palabras"].describe().reindex(CLASES).round(1))

# Filtrar outliers usando el percentil 99
tope = df["palabras"].quantile(0.99)

fig, ax = plt.subplots(figsize=(7, 3.5))
for clase in CLASES:
    grupo = df.loc[df["etiqueta"] == clase, "palabras"].clip(upper=tope)
    ax.hist(grupo, bins=15, alpha=.6, density=True, label=clase, color=COLORES.get(clase, "gray"))

ax.set_title(f"Tokens/Palabras por predio (proporción; recortado en percentil 99 = {tope:.0f})")
ax.set_xlabel("Cantidad de tokens")
ax.set_ylabel("Densidad")
ax.legend()
plt.tight_layout()
plt.show()

--- 4 · Longitud del texto descriptivo por clase -> escenas 5 y 6 ---
            count  mean  std  min  25%  50%  75%  max
etiqueta                                             
alto         85.0   4.0  0.5  3.0  4.0  4.0  4.0  5.0
medio       176.0   4.2  0.7  3.0  4.0  4.0  5.0  5.0
economico  1199.0   3.6  0.7  3.0  3.0  3.0  4.0  5.0


In [ ]:
print("--- 5 · Fechas por clase -> escena 7 (radiografía) ---")

# Reducir el dataset a 1,000 registros para evitar el colapso de RAM
df = df.head(1000).copy()

# Convertir la columna 'fecha' a formato datetime y extraer el año
df["anio"] = pd.to_datetime(df["fecha"], errors="coerce").dt.year

# Crear la tabla cruzada (crosstab)
tabla_anios = pd.crosstab(df["anio"], df["etiqueta"]).reindex(columns=CLASES, fill_value=0)
print(tabla_anios)

# Generar el gráfico de barras apiladas
tabla_anios.plot(
    kind="bar", 
    stacked=True, 
    figsize=(7, 3.5),
    color=[COLORES.get(c, "gray") for c in CLASES], 
    title="Distribución de Predios por Año y Clase (Muestra de 1000)"
)

plt.ylabel("Cantidad de predios")
plt.xlabel("Año")
plt.xticks(rotation=0)  
plt.tight_layout()
plt.show()